# dataset_123 train/test/val 분리

`dataset_123` 원본 이미지를 `data/train`, `data/test`, `data/val` 폴더로 클래스별 복사합니다.

- 비율: train 80%, test 10%, val 10%
- 원본 `dataset_123` 폴더는 삭제하거나 이동하지 않습니다.
- 같은 결과가 나오도록 `SEED`를 고정합니다.

In [1]:
from pathlib import Path
import random
import shutil
from collections import defaultdict

SOURCE_DIR = Path("dataset_123")
OUTPUT_DIR = Path("data")

TRAIN_RATIO = 0.8
TEST_RATIO = 0.1
VAL_RATIO = 0.1
SEED = 42

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# True이면 기존 data 폴더를 지우고 다시 만듭니다.
# 원본 dataset_123은 건드리지 않습니다.
RESET_OUTPUT = True

In [2]:
assert SOURCE_DIR.exists(), f"원본 폴더가 없습니다: {SOURCE_DIR}"
assert abs((TRAIN_RATIO + TEST_RATIO + VAL_RATIO) - 1.0) < 1e-9, "비율 합은 1이어야 합니다."

class_dirs = sorted([path for path in SOURCE_DIR.iterdir() if path.is_dir()])
assert class_dirs, f"클래스 폴더가 없습니다: {SOURCE_DIR}"

print("클래스 목록")
for class_dir in class_dirs:
    image_count = sum(1 for path in class_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    print(f"- {class_dir.name}: {image_count:,}장")

클래스 목록
- Attentive: 533장
- Drowsy: 586장
- LookingAway: 564장


In [3]:
def split_files(files, train_ratio=0.8, test_ratio=0.1, seed=42):
    files = list(files)
    rng = random.Random(seed)
    rng.shuffle(files)

    total = len(files)
    train_count = int(total * train_ratio)
    test_count = int(total * test_ratio)

    train_files = files[:train_count]
    test_files = files[train_count:train_count + test_count]
    val_files = files[train_count + test_count:]

    return {
        "train": train_files,
        "test": test_files,
        "val": val_files,
    }


split_plan = {}

for class_dir in class_dirs:
    image_files = sorted([
        path for path in class_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ])
    split_plan[class_dir.name] = split_files(
        image_files,
        train_ratio=TRAIN_RATIO,
        test_ratio=TEST_RATIO,
        seed=SEED,
    )

print("분리 예정 수량")
for class_name, splits in split_plan.items():
    train_count = len(splits["train"])
    test_count = len(splits["test"])
    val_count = len(splits["val"])
    total = train_count + test_count + val_count
    print(f"- {class_name}: train {train_count:,}, test {test_count:,}, val {val_count:,}, total {total:,}")

분리 예정 수량
- Attentive: train 426, test 53, val 54, total 533
- Drowsy: train 468, test 58, val 60, total 586
- LookingAway: train 451, test 56, val 57, total 564


In [4]:
if RESET_OUTPUT and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split_name in ["train", "test", "val"]:
    for class_name in split_plan:
        (OUTPUT_DIR / split_name / class_name).mkdir(parents=True, exist_ok=True)

copied_counts = defaultdict(int)

for class_name, splits in split_plan.items():
    for split_name, files in splits.items():
        target_dir = OUTPUT_DIR / split_name / class_name
        for src_path in files:
            dst_path = target_dir / src_path.name
            shutil.copy2(src_path, dst_path)
            copied_counts[(split_name, class_name)] += 1

print(f"완료: {OUTPUT_DIR.resolve()}")
for split_name in ["train", "test", "val"]:
    split_total = 0
    print(f"\n[{split_name}]")
    for class_name in sorted(split_plan):
        count = copied_counts[(split_name, class_name)]
        split_total += count
        print(f"- {class_name}: {count:,}장")
    print(f"합계: {split_total:,}장")

완료: C:\Projects\_active\focus_on_class\worktrees\3_compare\data

[train]
- Attentive: 426장
- Drowsy: 468장
- LookingAway: 451장
합계: 1,345장

[test]
- Attentive: 53장
- Drowsy: 58장
- LookingAway: 56장
합계: 167장

[val]
- Attentive: 54장
- Drowsy: 60장
- LookingAway: 57장
합계: 171장


In [5]:
print("생성된 폴더 확인")
for split_dir in sorted(OUTPUT_DIR.iterdir()):
    if not split_dir.is_dir():
        continue
    print(f"\n{split_dir}")
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            count = sum(1 for path in class_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
            print(f"  {class_dir.name}: {count:,}장")

생성된 폴더 확인

data\test
  Attentive: 53장
  Drowsy: 58장
  LookingAway: 56장

data\train
  Attentive: 426장
  Drowsy: 468장
  LookingAway: 451장

data\val
  Attentive: 54장
  Drowsy: 60장
  LookingAway: 57장
